<a href="https://colab.research.google.com/github/DerWeiseTeufel/yandexML_summerschool2026/blob/main/LLM_ScalingWeekTask2EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
from google.colab import userdata

# Load your CSV file

train_path = userdata.get('path_scalingWeek_train')

df = pd.read_csv(train_path)  # Replace with your actual file path
df.head()



,A,B,C,D,E,F,G,H,I,target
0,0.505,8,-,1.984,3.0,5,2.642,-5.122,0.649,1
1,0.536,4,-,1.977,1.0,3,5.756,-3.077,0.950,0
2,0.024,3,-,3.147,2.0,6,2.435,4.387,2.186,1
3,0.543,4,-,2.440,3.0,9,4.440,7.730,1.938,0
4,0.942,8,-,1.952,3.0,9,7.176,-4.579,0.346,1


In [ ]:
# Display basic information about the dataset
print("Dataset Shape:", df.shape)

Dataset Shape: (1000, 10)


In [ ]:
print("\nDataset Info:")
print(df.info())


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   A       1000 non-null   float64
 1   B       1000 non-null   int64  
 2   C       1000 non-null   object 
 3   D       1000 non-null   float64
 4   E       925 non-null    float64
 5   F       1000 non-null   int64  
 6   G       1000 non-null   float64
 7   H       1000 non-null   float64
 8   I       1000 non-null   float64
 9   target  1000 non-null   int64  
dtypes: float64(6), int64(3), object(1)
memory usage: 78.3+ KB
None


In [ ]:
print("\nDescriptive Statistics:")
print(df.describe())


Descriptive Statistics:
                 A            B            D           E            F  \
count  1000.000000  1000.000000  1000.000000  925.000000  1000.000000   
mean      0.504627     4.940000     2.038728    2.036757     5.578000   
std       0.317773     2.283772     0.346977    0.826214     2.877185   
min       0.000000     0.000000     1.649000    1.000000     1.000000   
25%       0.218750     3.000000     1.783000    1.000000     3.000000   
50%       0.511000     5.000000     1.944000    2.000000     6.000000   
75%       0.806500     6.000000     2.201250    3.000000     8.000000   
max       1.000000    15.000000     3.771000    3.000000    10.000000   

                 G            H            I       target  
count  1000.000000  1000.000000  1000.000000  1000.000000  
mean      5.064382    -0.156917     1.197345     0.288000  
std       1.600964     5.329461     0.770921     0.453058  
min       0.735000   -10.750000    -0.125000     0.000000  
25%       3.67475

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score
from sklearn.datasets import make_classification
from lightgbm import LGBMClassifier

In [ ]:
TARGET = "target"
N_FEATURES = 10
features = df.drop(labels=['target'], axis=1).columns.tolist()
cat_features = ['C' , 'E', "B", "F"]
real_features = [x for x in features if x not in cat_features]

In [ ]:
len(df[df['C']=="-"])

796

In [ ]:
features

['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I']

In [ ]:
df[features]

,A,B,C,D,E,F,G,H,I
0,0.505,8,-,1.984,3.0,5,2.642,-5.122,0.649
1,0.536,4,-,1.977,1.0,3,5.756,-3.077,0.950
2,0.024,3,-,3.147,2.0,6,2.435,4.387,2.186
3,0.543,4,-,2.440,3.0,9,4.440,7.730,1.938
4,0.942,8,-,1.952,3.0,9,7.176,-4.579,0.346
...,...,...,...,...,...,...,...,...,...
995,0.985,3,-,2.233,1.0,8,5.876,6.461,1.836
996,0.369,5,-,1.947,1.0,3,4.922,3.857,2.213
997,0.454,11,-,2.184,1.0,3,7.198,9.365,2.587
998,0.012,7,-,1.832,1.0,8,5.290,3.528,2.364


In [ ]:
# 1. Correlation Heatmap
corr_matrix = df[real_features].corr()
fig_corr = px.imshow(corr_matrix,
                    title="Feature Correlation Heatmap",
                    color_continuous_scale='RdBu_r',
                    aspect="auto")
fig_corr.show()


I and A are quite linear correlating

In [ ]:
df['E'].value_counts()

,count
E,
3.0,333
1.0,299
2.0,293


In [ ]:
df['B'].value_counts()

,count
B,
5,185
4,175
6,145
3,140
2,92
7,91
8,54
1,39
9,37


In [ ]:
df['F'].value_counts()

,count
F,
5,109
7,107
9,105
10,103
8,101
1,99
2,97
4,95
3,92


In [ ]:

# 1. Distribution of Target Variable
fig1 = px.pie(df, names='target', title='Distribution of Target Variable')
fig1.show()

In [ ]:

continuous_features = cat_features
# Create subplots for continuous features
fig2 = make_subplots(
    rows=4, cols=2,
    subplot_titles=continuous_features,
    vertical_spacing=0.08
)

for i, feature in enumerate(continuous_features, 1):
    row = (i-1)//2 + 1
    col = (i-1)%2 + 1

    fig2.add_trace(
        go.Histogram(x=df[feature], name=feature, nbinsx=50),
        row=row, col=col
    )

fig2.update_layout(
    height=800,
    title_text="Distribution of Continuous Features",
    showlegend=False
)
fig2.show()

In [ ]:
continuous_features = real_features
# 3. Box plots for continuous features by target
fig3 = make_subplots(
    rows=4, cols=2,
    subplot_titles=[f'{feature} by Target' for feature in continuous_features],
    vertical_spacing=0.08
)

for i, feature in enumerate(continuous_features, 1):
    row = (i-1)//2 + 1
    col = (i-1)%2 + 1

    for target_value in df['target'].unique():
        fig3.add_trace(
            go.Box(
                y=df[df['target'] == target_value][feature],
                name=f'Target {target_value}',
                legendgroup=target_value,
                showlegend=(i == 1)  # Only show legend for first subplot
            ),
            row=row, col=col
        )

fig3.update_layout(
    height=800,
    title_text="Continuous Features Distribution by Target",
    boxmode='group'
)
fig3.show()

ValueError: 
    Invalid value of type 'numpy.int64' received for the 'legendgroup' property of box
        Received value: np.int64(1)

    The 'legendgroup' property is a string and must be specified as:
      - A string
      - A number that will be converted to a string

In [ ]:
# 3. Box plots for continuous features by target - FIXED
fig3 = make_subplots(
    rows=4, cols=2,
    subplot_titles=[f'{feature} by Target' for feature in continuous_features],
    vertical_spacing=0.08
)

for i, feature in enumerate(continuous_features, 1):
    row = (i-1)//2 + 1
    col = (i-1)%2 + 1

    for target_value in df['target'].unique():
        fig3.add_trace(
            go.Box(
                y=df[df['target'] == target_value][feature],
                name=f'Target {target_value}',
                legendgroup=str(target_value),  # Convert to string
                showlegend=(i == 1)  # Only show legend for first subplot
            ),
            row=row, col=col
        )

fig3.update_layout(
    height=800,
    title_text="Continuous Features Distribution by Target",
    boxmode='group'
)
fig3.show()

In [ ]:


# 4. Distribution of Binary Feature C by Target
fig4 = px.histogram(df, x='C', color='target', barmode='group',
                   title='Binary Feature C Distribution by Target',
                   labels={'C': 'Binary Feature C', 'count': 'Count'})
fig4.show()

# 5. Correlation Heatmap (including encoded binary feature)
# Encode binary feature C for correlation
df_encoded = df.copy()
df_encoded['C_encoded'] = df_encoded['C'].map({'-': 0, '+': 1})

# Select only numeric columns for correlation
numeric_cols = continuous_features + ['C_encoded', 'target']
correlation_matrix = df_encoded[numeric_cols].corr()

fig5 = px.imshow(correlation_matrix,
                title='Correlation Heatmap',
                color_continuous_scale='RdBu_r',
                aspect='auto')
fig5.show()


In [ ]:
# 6. Scatter plots for feature relationships colored by target
# Create scatter matrix for continuous features
fig6 = px.scatter_matrix(df,
                        dimensions=continuous_features[:4],  # First 4 features to avoid overcrowding
                        color='target',
                        title='Scatter Matrix of Continuous Features by Target',
                        height=800)
fig6.show()




In [ ]:
# 7. Individual scatter plots with trend lines
fig7 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'{continuous_features[0]} vs {feature}' for feature in continuous_features[1:5]],
    vertical_spacing=0.1
)

for i, feature in enumerate(continuous_features[1:5], 1):
    row = (i-1)//2 + 1
    col = (i-1)%2 + 1

    for target_value in df['target'].unique():
        scatter_data = df[df['target'] == target_value]
        fig7.add_trace(
            go.Scatter(
                x=scatter_data[continuous_features[0]],
                y=scatter_data[feature],
                mode='markers',
                name=f'Target {target_value}',
                legendgroup=target_value,
                showlegend=(i == 1)
            ),
            row=row, col=col
        )

fig7.update_layout(
    height=600,
    title_text=f"Scatter Plots with {continuous_features[0]}"
)
fig7.show()

ValueError: 
    Invalid value of type 'numpy.int64' received for the 'legendgroup' property of scatter
        Received value: np.int64(1)

    The 'legendgroup' property is a string and must be specified as:
      - A string
      - A number that will be converted to a string

In [ ]:


# 8. Violin plots for continuous features by target
fig8 = make_subplots(
    rows=4, cols=2,
    subplot_titles=continuous_features,
    vertical_spacing=0.08
)

for i, feature in enumerate(continuous_features, 1):
    row = (i-1)//2 + 1
    col = (i-1)%2 + 1

    for target_value in df['target'].unique():
        fig8.add_trace(
            go.Violin(
                x=[target_value] * len(df[df['target'] == target_value]),
                y=df[df['target'] == target_value][feature],
                name=f'Target {target_value}',
                legendgroup=target_value,
                showlegend=(i == 1),
                side='negative' if target_value == df['target'].unique()[0] else 'positive'
            ),
            row=row, col=col
        )

fig8.update_layout(
    height=800,
    title_text="Violin Plots of Continuous Features by Target",
    violinmode='overlay'
)
fig8.show()


ValueError: 
    Invalid value of type 'numpy.int64' received for the 'legendgroup' property of violin
        Received value: np.int64(1)

    The 'legendgroup' property is a string and must be specified as:
      - A string
      - A number that will be converted to a string

In [ ]:

# 9. Statistical summary
print("\nStatistical Summary:")
print(df[continuous_features + ['target']].describe())

# 10. Missing values visualization
if df.isnull().sum().sum() > 0:
    missing_data = df.isnull().sum()
    missing_data = missing_data[missing_data > 0]

    fig10 = px.bar(x=missing_data.index, y=missing_data.values,
                  title='Missing Values by Feature',
                  labels={'x': 'Features', 'y': 'Missing Count'})
    fig10.show()

# 11. Interactive pairplot with target
fig11 = px.scatter(df, x=continuous_features[0], y=continuous_features[1],
                  color='target', size=continuous_features[2] if len(continuous_features) > 2 else None,
                  hover_data=df.columns,
                  title=f'{continuous_features[0]} vs {continuous_features[1]} colored by Target')
fig11.show()

print("\nEDA Completed! Check all the interactive plots above.")


Statistical Summary:
                 A            D            G            H            I  \
count  1000.000000  1000.000000  1000.000000  1000.000000  1000.000000   
mean      0.504627     2.038728     5.064382    -0.156917     1.197345   
std       0.317773     0.346977     1.600964     5.329461     0.770921   
min       0.000000     1.649000     0.735000   -10.750000    -0.125000   
25%       0.218750     1.783000     3.674750    -5.031750     0.546500   
50%       0.511000     1.944000     5.483000    -1.220500     1.103000   
75%       0.806500     2.201250     6.249500     4.925000     1.644250   
max       1.000000     3.771000     8.375000    11.475000     4.279000   

            target  
count  1000.000000  
mean      0.288000  
std       0.453058  
min       0.000000  
25%       0.000000  
50%       0.000000  
75%       1.000000  
max       1.000000  



EDA Completed! Check all the interactive plots above.
